Report 5


Anh Do

020416-2317

anhd@kth.se

In [268]:
import matplotlib.pyplot as plt
import pyomo.environ as pyo
import numpy as np
from gurobipy import GRB
import gurobipy as gp
from pyomo.opt import SolverFactory

WLS = { # Please dont steal my credentials
    "WLSACCESSID": "1e6bdedb-f27d-4b0c-8a0d-24a8c94a8cb5",
    "WLSSECRET": "47ebeda5-b872-4365-9981-ef8044c28db5",
    "LICENSEID": 2707350,
}

# Problem 1

## Code

### ECP

In [269]:
def build_ecp_model():
    m = pyo.ConcreteModel()
    m.I = pyo.RangeSet(1, 8)

    def bounds_rule(m, i):
        if i >= 5: 
            return (0, 3)
        return (-2, 2)
    
    def domain_rule(m, i):
        if i >= 5:
            return pyo.Integers
        return pyo.Reals

    m.x = pyo.Var(m.I, bounds=bounds_rule, domain=domain_rule)
    m.obj = pyo.Objective(expr=-sum(m.x[i] for i in m.I), sense=pyo.minimize)
    
    # Linear constraint
    m.linear_const = pyo.Constraint(expr=m.x[1] + m.x[3] + m.x[5] + m.x[7] <= 2)
    
    # Container for ECP cuts
    m.ecp_cuts = pyo.ConstraintList()

    return m

In [270]:
grb_params = {"TimeLimit": 300, "MIPFocus": 1, "OutputFlag": 0}

with pyo.SolverFactory('gurobi_persistent', manage_env=True) as solver:
    solver.set_options(WLS)
    solver.options.update(grb_params)
    
    model = build_ecp_model()
    solver.set_instance(model)
    
    tol = 1e-4
    max_iter = 1000
    iteration = 0

    print(f"{'Iter':<5} | {'Obj Value':<12} | {'Violation':<12}")
    print("-" * 40)

    while iteration < max_iter:
        iteration += 1
        
        # 1. Solve Master
        solver.solve(model)
        
        # 2. Get current values
        x_val = {i: pyo.value(model.x[i]) for i in model.I}
        current_obj = pyo.value(model.obj)
        
        # 3. Check Constraint Violation: sum(x^2) - 3 <= 0
        sum_sq = sum(x_val[i]**2 for i in model.I)
        g_val = sum_sq - 3
        
        if g_val <= tol:
            print(f"{iteration:<5} | {current_obj:<12.6f} | {g_val:<12.6f} (Converged!)")
            break
        
        print(f"{iteration:<5} | {current_obj:<12.6f} | {g_val:<12.6f}")
        
        # 4. Add Cut: g(x^k) + grad * (x - x^k) <= 0
        # g(x^k) = g_val
        # grad = 2 * x_val
        lhs = g_val + sum(2 * x_val[i] * (model.x[i] - x_val[i]) for i in model.I)
        
        model.ecp_cuts.add(lhs <= 0)
        solver.add_constraint(model.ecp_cuts[len(model.ecp_cuts)])

    print("-" * 40)
    print("Final Solution:")
    for i in model.I:
        print(f"x[{i}] = {pyo.value(model.x[i]):.6f}")

Iter  | Obj Value    | Violation   
----------------------------------------
1     | -12.000000   | 27.000000   
2     | -10.000000   | 27.000000   
3     | -9.583333    | 24.395833   
4     | -8.750000    | 13.062500   
5     | -8.750000    | 23.062500   
6     | -8.483333    | 16.076445   
7     | -7.950000    | 9.279863    
8     | -7.471173    | 13.059594   
9     | -7.301533    | 6.996698    
10    | -6.666427    | 6.375741    
11    | -6.426148    | 10.203227   
12    | -6.413840    | 3.738519    
13    | -6.275229    | 4.524293    
14    | -6.132209    | 7.552628    
15    | -6.000000    | 7.950853    
16    | -5.835908    | 3.137654    
17    | -5.823421    | 10.197482   
18    | -5.778358    | 4.701756    
19    | -5.755645    | 2.972649    
20    | -5.728472    | 3.433632    
21    | -5.530254    | 2.720442    
22    | -5.389863    | 3.220922    
23    | -5.387168    | 3.613622    
24    | -5.337083    | 1.650179    
25    | -5.293258    | 1.706800    
26    | -5.195933    | 

### OA

In [271]:
def create_nlp_model(y_fixed):
    m = pyo.ConcreteModel()
    m.I = pyo.RangeSet(1, 8)

    # Integer intersection bounds {0,1,2,3}
    def bounds_rule(m, i):
        if i >= 5: 
            return (0, 3) 
        return (-2, 2)
    
    def domain_rule(m, i):
        if i >= 5: 
            return pyo.Integers
        return pyo.Reals
    
    m.x = pyo.Var(m.I, bounds=lambda m, i: bounds_rule(m, i), domain=lambda m, i: domain_rule(m, i))
    
    m.obj = pyo.Objective(expr=-sum(m.x[i] for i in m.I), sense=pyo.minimize)
    
    for i in range(5, 9):
        m.x[i].fix(y_fixed[i])

    # Constraints
    m.lincon = pyo.Constraint(expr=m.x[1] + m.x[3] + m.x[5] + m.x[7] - 2 <= 0)
    m.concon = pyo.Constraint(expr=sum(m.x[i]**2 for i in m.I) - 3 <= 0)

    return m

def create_feas_model(y_fixed):
    m = pyo.ConcreteModel()
    m.I = pyo.RangeSet(1, 8)

    # Integer intersection bounds {0,1,2,3}
    def bounds_rule(m, i):
        if i >= 5: 
            return (0, 3) 
        return (-2, 2)
    
    def domain_rule(m, i):
        if i >= 5: 
            return pyo.Integers
        return pyo.Reals
    
    m.x = pyo.Var(m.I, bounds=lambda m, i: bounds_rule(m, i), domain=lambda m, i: domain_rule(m, i))

    m.u = pyo.Var(domain=pyo.NonNegativeReals)
    m.obj = pyo.Objective(expr=m.u, sense=pyo.minimize)
    
    for i in range(5, 9):
        m.x[i].fix(y_fixed[i])
        
    # Relaxed Constraints
    m.lincon = pyo.Constraint(expr=m.x[1] + m.x[3] + m.x[5] + m.x[7] - 2 <= 0)

    m.concon = pyo.Constraint(expr=sum(m.x[i]**2 for i in m.I) - 3 <= m.u)
    return m

def create_master_model():
    m = pyo.ConcreteModel()
    m.I = pyo.RangeSet(1, 8)
    
    # Integer intersection bounds {0,1,2,3}
    def bounds_rule(m, i):
        if i >= 5: return (0, 3) 
        return (-2, 2)
    
    def domain_rule(m, i):
        if i >= 5: return pyo.Integers
        return pyo.Reals

        
    m.mu = pyo.Var(domain=pyo.Reals)
    m.x = pyo.Var(m.I, bounds=lambda m, i: bounds_rule(m, i), domain=lambda m, i: domain_rule(m, i))
    
    # Objective: Minimize mu
    m.obj = pyo.Objective(expr=m.mu, sense=pyo.minimize)

    # Linear Objective Equivalent to Original
    m.linobj = pyo.Constraint(expr=m.mu >= -sum(m.x[i] for i in m.I))

    # My lazy cut so i dont need to handle vectorized constraints that is always the strongest (static cut) because we have linearity
    m.static_cut = pyo.Constraint(expr=m.x[1] + m.x[3] + m.x[5] + m.x[7] - 2 <= 0)
    
    # Container for all nonlinear constraint linearizations (both feasible and infeasible cases)
    m.nonlinear_cuts = pyo.ConstraintList()
    
    return m

In [ ]:
grb_params = {"TimeLimit": 300, "MIPFocus": 1, "OutputFlag": 0}

with pyo.SolverFactory('gurobi_persistent', manage_env=True) as master_solver:
    master_solver.set_options(WLS)
    master_solver.options.update(grb_params)
    master_model = create_master_model()
    master_solver.set_instance(master_model)
    
    max_iter = 10
    iteration = 0
    guess = {5: 0, 6: 0, 7: 0, 8: 0} # Initial Integer Guess
    prev_guess = None

    print(f"{'Iter':<5} | {'Obj Value':<12} | {'Status':<12} | {'x_vals':<20} | {'Other':<10}")
    print("-" * 60)

    while iteration < max_iter:
        iteration += 1
        
        prev_guess = guess.copy()
        
        # --- 1. Solve NLP Subproblem ---
        NLP_model = create_nlp_model(guess)
        NLP_solver = SolverFactory('gurobi')
        result_nlp = NLP_solver.solve(NLP_model, load_solutions=False)

        if result_nlp.solver.termination_condition == pyo.TerminationCondition.optimal:
            NLP_model.solutions.load_from(result_nlp)
            f_k = pyo.value(NLP_model.obj)
            x_k = {i: pyo.value(NLP_model.x[i]) for i in NLP_model.I}
            g_k = sum(x_k[i]**2 for i in NLP_model.I) - 3
            
            print(f"{iteration:<5} | {f_k:<12.4f} | {'Feasible':<12} | {str(x_k):<20} | {f_k:<10}")

            if f_k < master_model.mu.ub if master_model.mu.ub is not None else float('inf'):
                master_model.mu.setub(f_k)
                x_opt = x_k.copy()

            if NLP_model.concon.active:
                new_cons = master_model.nonlinear_cuts.add(
                    expr= 0 >= g_k + sum([(2*x_k[i])*(master_model.x[i] - x_k[i]) for i in master_model.I])
                )
                master_solver.add_constraint(new_cons)

        elif result_nlp.solver.termination_condition == pyo.TerminationCondition.infeasible:           
            # --- 2. Solve Feasibility Problem (Relaxation) ---
            FEAS_model = create_feas_model(guess)
            FEAS_solver = SolverFactory('gurobi_direct')
            result_feas = FEAS_solver.solve(FEAS_model, load_solutions=False)
            FEAS_model.solutions.load_from(result_feas)
    
            x_k = {i: pyo.value(FEAS_model.x[i]) for i in FEAS_model.I}
            g_k = sum((x_k[i]**2 for i in FEAS_model.I)) - 3

            print(f"{iteration:<5} | {'--':<12} | {'Infeasible':<12} | {str(x_k):<20} | {pyo.value(FEAS_model.u):<10}")

            # Add Infeasibility Cut: g(x^k) + ∇g(x^k)^T (x - x^k) <= 0
            if FEAS_model.concon.active:
                new_cons = master_model.nonlinear_cuts.add(
                    expr= 0 >= g_k + sum([(2*x_k[i])*(master_model.x[i] - x_k[i]) for i in master_model.I])
                )
                master_solver.add_constraint(new_cons)

        else:
            raise RuntimeError(f"Unexpected NLP status: {result_nlp.solver.termination_condition}")


        # --- 3. Solve Master Problem ---
        result_master = master_solver.solve()
        x_k = {i: pyo.value(master_model.x[i]) for i in master_model.I}

        print(f"{iteration:<5} | {pyo.value(master_model.mu):<12.4f} | {'MASTER':<12} | {str(x_k):<20}")

        if result_master.solver.termination_condition == pyo.TerminationCondition.infeasible:
            print("Master Infeasible -> Last solution was optimal.")
            break
        
        # --- 4. Update Integer Guess ---
        # Rounding is critical for integer vars returned by solvers
        #print(master_model.mu.pprint())
        guess = {i: (round(pyo.value(master_model.x[i]))) for i in range(5, 9)}
    
    print("-" * 60)
    print("Final Solution:")
    print(f"Objective: {pyo.value(master_model.mu.ub):.6f}")
    for i in master_model.I:
        print(f"x[{i}] = {x_opt[i]}")

Iter  | Obj Value    | Status       | x_vals               | Other     
------------------------------------------------------------
1     | -3.4641      | Feasible     | {1: 0.8661067688905831, 2: 0.8659435622023987, 3: 0.8661067688905831, 4: 0.8659435622023987, 5: 0, 6: 0, 7: 0, 8: 0} | -3.4641006621859636
1     | -12.0000     | MASTER       | {1: -2.0, 2: 2.0, 3: -2.0, 4: 2.0, 5: 3.0, 6: 3.0, 7: 3.0, 8: 3.0}
2     | --           | Infeasible   | {1: -2.0, 2: -0.0, 3: -2.0, 4: -0.0, 5: 3, 6: 3, 7: 3, 8: 3} | 41.0      
2     | -10.4645     | MASTER       | {1: 1.4645289963894526, 2: 2.0, 3: -2.0, 4: 2.0, 5: 2.0, 6: 2.0, 7: -0.0, 8: 3.0}
3     | --           | Infeasible   | {1: -1.875358924419146e-05, 2: -0.0, 3: -1.875358924419146e-05, 4: -0.0, 5: 2, 6: 2, 7: 0, 8: 3} | 14.000000000703395
3     | -10.0000     | MASTER       | {1: 1.0, 2: 2.0, 3: -2.0, 4: 2.0, 5: 0.0, 6: 3.0, 7: 3.0, 8: 1.0}
4     | --           | Infeasible   | {1: -0.5, 2: -0.0, 3: -0.5, 4: -0.0, 5: 0, 6: 3, 7: 3, 

In [273]:
NLP_model.pprint()

1 RangeSet Declarations
    I : Dimen=1, Size=8, Bounds=(1, 8)
        Key  : Finite : Members
        None :   True :   [1:8]

1 Var Declarations
    x : Size=8, Index=I
        Key : Lower : Value : Upper : Fixed : Stale : Domain
          1 :    -2 :   0.0 :     2 : False :  True :    Reals
          2 :    -2 :   0.0 :     2 : False :  True :    Reals
          3 :    -2 :   0.0 :     2 : False :  True :    Reals
          4 :    -2 :   0.0 :     2 : False :  True :    Reals
          5 :     0 :     1 :     3 :  True :  True : Integers
          6 :     0 :     1 :     3 :  True :  True : Integers
          7 :     0 :     1 :     3 :  True :  True : Integers
          8 :     0 :     0 :     3 :  True :  True : Integers

1 Objective Declarations
    obj : Size=1, Index=None, Active=True
        Key  : Active : Sense    : Expression
        None :   True : minimize : - (x[1] + x[2] + x[3] + x[4] + x[5] + x[6] + x[7] + x[8])

2 Constraint Declarations
    concon : Size=1, Index=Non